# Solar Filament Segmentation Challenge 2026

## 1. Competition Introduction & Data Testbed

* **Core Phenomenon**: Solar filaments are dense, cold plasma structures suspended by magnetic field lines in the solar corona. They are the catalyst for space weather eruptions like Coronal Mass Ejections (CMEs), which can catastrophically **damage Earth's power grids, GPS, and communication infrastructure**.
* **MAGFiLO Dataset**: Standing for Manually Annotated GONG Filaments from H-Alpha Observations, this dataset functions as the official benchmarking testbed.
* **Data Constraints**: Models must exclusively utilize the provided H-alpha test images for inference and are **strictly forbidden from training with any outside ground-truth metadata**.

## 2. Evaluation Rubric & Metrics
The final score combines a quantitative performance review and a qualitative pipeline assessment:

* **Quantitative Comparison (70%)**: Evaluates overlapping pixel accuracy, geometric distribution, error handling, and runtime mechanics:
* **Panoptic Quality (PQ)**: The primary leaderboard ranking metric, calculating combined instance detection and segmentation overlap.
   * **Mean Dice Score & IoU**: Tracked via `torchmetrics.segmentation.DiceScore` to evaluate overlap accuracy.
   * **Structural Mappings**: Tracks distribution of one-to-many (fragmentation) and many-to-one (over-merging) prediction faults.
* **Qualitative Evaluation (30%)**: Assesses end-to-end pipeline descriptions, structural morphology accuracy on H-alpha imagery, and code modularity/documentation.
* **Scoreboard Splits**: The public scoreboard tracks the mean Dice score for **~50% of the test set images**. The remaining **50% stays hidden** for internal organizer validation.

## 3. Technical Implementation Challenges
Algorithms must be engineered to overcome three distinct physical and visual obstacles:

* **Fine-scale Structures**: Difficulty capturing **barbs** (thin, thread-like filament extensions) which dictate the underlying magnetic field orientation.
* **Background Noise**: Differentiating dark filament shapes from solar background artifacts, sunspots, and ground-based observatory imaging noise.
* **Structural Continuity**: Preventing models from over-segmenting continuous structures into broken pieces ("islands") or over-merging independent adjacent structures.

## 4. Submission Specifications

* **File Format**: A single **CSV file** detailing the entire test set.
* Required Columns:
* `filament_id`: A unique ID string appended with a distinct tail string marker per instance (e.g., `_1`, `_2`, `_3`) for images hosting multiple filaments.
   * `segmentation_rle`: Unquoted **Run-Length Encoded (RLE) Counts** representing the segmentation mask.
* **Dimension Guidelines**: All processing masks map to a fixed resolution layout of **2048 x 2048 pixels**. Predictions are evaluated based on coordinate overlap rather than index order.
* **Conversion Tools**: Competitors must format masks to RLE counts via the pycocotools Python library (`annToMask`, `decodeMask`, `encodeMask`).

## 5. Prizes & Post-Competition Requirements
A total prize pool of up to **$3,000** will be divided among the top **3 teams**. Individual prize allocations are calculated using a weighted temperature formula (T=2.0) based on the evaluation rubric scores.
Winners will be announced at **IEEE BigData 2026 (December 14–17, 2026, Phoenix, AZ)** contingent on meeting the following requirements:

* **Open-Access Code Release**: A publicly accessible Git repository hosting all scripts, weights, and configurations must be opened immediately at competition close. The source code must seamlessly reproduce the submitted leaderboard results.
* **Technical Report**: Submission of a concise technical layout documenting the preprocessing, architecture, training configurations, and post-processing strategies.
* **Administrative Setup**: Compliance with AURA onboarding vendors, providing direct banking information capable of receiving transfers from U.S. financial institutions.






# Data and Directory Structure

## 1. Directory Structure & Data Constraints
When unzipped, the `MAGFiLO_1.0_Kaggle_2026/` directory contains the following structural layout:

* **train/**: Contains `train_images/` and the master annotation file `MAGFiLO_1.0_Annotations_kaggle2026_train.json`.
* **test/**: Contains `test_images/` used exclusively for model inference. No outside ground-truth metadata can be applied during training.

## 2. Image Properties & Naming Convention

* **Format**: Custom solar physics FITS files have been converted into standard **8-bit grayscale JPEG images** with a **fixed resolution of 2048 x 2048 pixels**. They **must be processed purely as grayscale (not multi-channel color/RGB)**.
* **Filename Metadata Syntax**: Coded as `<YYYYMMDDHHMMSSII>.jpeg` where II denotes the ground-based observatory station.
  * Example: `20260901165702Bh.jpeg` marks a capture on Sept 1, 2026, at 16:57:02 by the **Big Bear (Bh)** Solar Observatory.
* **Annotator Inter-Variability**: Multiple expert annotators independently reviewed the same raw files. If the same image features two distinct string prefixes (e.g., `010101-...` and `010102-...`), you should **treat them simply as separate, independent training samples**.

## 3. JSON COCO-Style Annotation Anatomy

The training annotations use the standardized COCO data format profile, making them fully compatible with the `pycocotools` framework.

### `image` Node Fields

* `id`: A unique string merging the annotator's batch ID and the target filename (e.g., `010401-20160920230134Lh`).
* `file_name`: String file path pointer mapping back to the images folder.

### `annotation` Node Fields

* `id`: A unique UUID string identifying individual filament entries.
* `image_id`: String mapping directly back to the image["id"].
* `category_id`: Integer tracking structural classification boundaries.
* `segmentation`: A list containing **exactly one single-piece polygon** mapped as a sequence of vertex coordinates: `[x0, y0, x1, y1, ..., xn-1, yn-1]`.
* `spine`: A contiguous coordinate path charting the primary structural "backbone" of the solar filament.
* `bbox`: Four-element pixel coordinates measuring `[x, y, width, height]` from the top-left corner.
* `iscrowd`: Always set to 0, explicitly dictating that no multi-filament blending clusters exist.

### `categories` Reference Map

Filaments are categorically grouped into four distinct target states based on structural morphology:

* `1`: "Left"
* `2`: "Right"
* `3`: "Unidentifiable"
* `4`: "Ambiguous"

## 4. Technical Submission & Validation Strategy

* **Format Specification**: The output CSV demands a `filament_id` alongside a `segmentation_rle` column populated by unquoted **Run-Length Encoded (RLE)** Counts.
* **Instance Splitting**: If an algorithm detects 3 isolated filaments in an image file, the final CSV must contain 3 rows uniquely designated with incremental trailing tags (e.g., `_1`, `_2`, `_3`).
* **Evaluation Targets**: Submissions are checked using **Panoptic Quality (PQ)** and the **Dice Score** via coordinate overlap matching. **Fragmented networks ("islands") or over-merged structures are heavily penalized**.
* **Winner Verification**: Top-3 payouts (from a total pool of **$3,000**) require open-sourcing the solution under an OSI-approved license alongside a detailed technical pipeline report at **IEEE BigData 2026**.


# Load Data and Sanity Check

In this section, we will load the MAGFiLO dataset and perform a sanity check to ensure that the data is correctly formatted and accessible. We will also visualize a few sample images along with their corresponding annotations to get an initial understanding of the dataset.

## 1. Import Libraries

In [18]:
# =========================
# 1. Import Libraries
# =========================
import os
import json
import warnings
from importlib import metadata
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from matplotlib import pyplot as plt
from PIL import Image
from pycocotools.coco import COCO

# Suppress annoying background warnings
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# Set random seed for reproducibility
RANDOM_SEED = 10

# Change the path to your dataset directory
# Kaggle
# INPUT_ROOT = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
INPUT_ROOT = Path("D:/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

sns.set_theme(style="whitegrid", palette="muted", context="notebook")

print(f"NumPy:      {metadata.version('numpy')}")
print(f"Pandas:     {metadata.version('pandas')}")
print(f"Matplotlib: {metadata.version('matplotlib')}")
print(f"Seaborn:    {metadata.version('seaborn')}")
print(f"TensorFlow: {metadata.version('tensorflow')}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Input root:  {INPUT_ROOT}")

NumPy:      2.5.3
Pandas:     3.0.6
Matplotlib: 3.11.2
Seaborn:    0.13.2
TensorFlow: 2.21.0
Random seed: 10
Input root:  D:\filament-segmentation-2026\MAGFiLO_1.0_Kaggle_2026


## 2. Describe Dataset Structure

In [19]:
if not INPUT_ROOT.exists():
    raise FileNotFoundError(f"Input root directory '{INPUT_ROOT}' does not exist.")

def walk_and_describe(current_path: Path, root_path: Path) -> list:
    """Recursively traverses directories to count immediate files and format paths."""
    results = []
    
    # Separate immediate children into files and folders
    immediate_items = list(current_path.iterdir())
    files_count = sum(1 for item in immediate_items if item.is_file())
    subfolders = [item for item in immediate_items if item.is_dir()]
    
    # Format path relative to the root (e.g., 'Folder/folder')
    relative_path = current_path.relative_to(root_path.parent)
    display_name = relative_path.as_posix() 
    
    # 1. Append current directory information
    results.append({
        "name": display_name,
        "type": "directory",
        "size_mb": np.nan,
        "children": files_count,
    })
    
    # 2. Recursively descend into nested folders
    for folder in subfolders:
        results.extend(walk_and_describe(folder, root_path))
        
    return results

all_rows = walk_and_describe(INPUT_ROOT, INPUT_ROOT)
input_structure = pd.DataFrame(all_rows)

display(input_structure)


,name,type,size_mb,children
0,MAGFiLO_1.0_Kaggle_2026,directory,NaN,0
1,MAGFiLO_1.0_Kaggle_2026/test,directory,NaN,0
2,MAGFiLO_1.0_Kaggle_2026/test/test_images,directory,NaN,180
3,MAGFiLO_1.0_Kaggle_2026/train,directory,NaN,1
4,MAGFiLO_1.0_Kaggle_2026/train/train_images,directory,NaN,707


## i.Observation:

There are 2 main folders in the dataset: `train` and `test`. The `train` folder contains json annotations and 707 images, while the `test` folder contains 180 images. A split of 79.1% train and 20.9% test images.

## 3. Load Json Annotations

In [27]:
# =========================
# 3. Load Json Annotations
# =========================

annotation_file = os.path.join(INPUT_ROOT, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")

with open(annotation_file, "r") as f:
    annotations_data = json.load(f)
    
print(annotations_data.keys())
print(annotations_data["info"])

dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])
{'description': 'MLEcoFi 2024 Dataset - V1.0', 'url': 'https://www.mlecofi.net/', 'version': '1.0', 'year': 2024, 'contributor': 'MLEcoFi Team', 'date_created': '2024-07-15', 'team': 'Earth-Space AI Research Lab (www.esairlab.com)', 'collaborators': ['National Solar Observatory, Georgia State University'], 'date_updated': '2026-06-01'}


# EDA(Exploratory Data Analysis)

In this section, we will perform an exploratory data analysis (EDA) on the MAGFiLO dataset to understand the characteristics of the solar filament images and their annotations. This will help us identify patterns, anomalies, and potential preprocessing steps for our segmentation model.